# Context and State

Context: specific and relevant information an agent has about a question or task that will improve the accuracy and reliability of its answers
State: the conditions the agent starts the task from

In [20]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool, ToolRuntime

from dotenv import load_dotenv

from dataclasses import dataclass

## Context

Context is only passed during tool calls at runtime, not when the agent is configured. Therefore, adding context to the agent requires four steps:
- Creating a dataclass with the appropriate context
- Setting up tool calls to utilize or retrieve context
- Pass the context schema to the agent when it is created
- Pass an instance of the context class to the agent when it is invoked

#### Dataclass

In [14]:
@dataclass
class ColorContext:
    favorite_color: str = "blue"
    least_favorite_color: str = "orange"

#### Tools

In [23]:
@tool
def get_favorite_color(runtime: ToolRuntime) -> str:
    """Get the favorite color of the user"""
    return runtime.context.favorite_color

@tool
def get_least_favorite_color(runtime: ToolRuntime) -> str:
    """Get the least favoriate color of the user"""
    return runtime.context.least_favorite_color

#### Creating the Agent

In [24]:
agent = create_agent(
    model='claude-haiku-4-5',
    context_schema=ColorContext,
    tools=[get_favorite_color, get_least_favorite_color]
)

#### Implementing

In [25]:
msg = HumanMessage(content="What is my least favorite color?")

response = agent.invoke({'messages': [msg]},
                        context=ColorContext(),
                        tools=[get_favorite_color, get_least_favorite_color])

/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favorite_col...favorite_color='orange'), input_type=ColorContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favorite_col...favorite_color='orange'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


In [26]:
response

{'messages': [HumanMessage(content='What is my least favorite color?', additional_kwargs={}, response_metadata={}, id='d2c06613-667e-4958-9847-e1e38389d840'),
  AIMessage(content=[{'id': 'toolu_01B7JT4pYg6y9GoLA3fQPWf1', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_least_favorite_color', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ce6jKG5fwzFGXVKffbXBU', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 600, 'output_tokens': 41, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a00b5e-6196-7c80-b6d6-eaf8a3122245-0', tool_calls=[{'name': 'g

## State